# Steering Runs

In [1]:
!pip install -q transformers accelerate h5py huggingface_hub scikit-learn python-dotenv
!pip install -q anthropic
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## Setup — Drive, HF login, repo

In [2]:
from pathlib import Path
import os, sys
from google.colab import drive
from dotenv import load_dotenv

drive.mount('/content/drive', force_remount=True)
from huggingface_hub import login

load_dotenv("/content/drive/MyDrive/.secrets/hf.env")
hf_token = os.getenv("HF_TOKEN")
assert hf_token is not None, "HF_TOKEN not found"
login(token=hf_token)

In [3]:
REPO_DIR = Path("/content/emotion-mechanisms-llm")
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/daspushpita/emotion-mechanisms-llm.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

for p in [f'{REPO_DIR}/src', f'{REPO_DIR}/scripts']:
    if p not in sys.path:
        sys.path.insert(0, p)

Already up to date.


## Configuration

In [5]:
ROOT_PATH    = Path("/content/drive/MyDrive/emotion-mechanisms-llm")
DATA_PATH    = ROOT_PATH / "qwen_v2"

LAYERS         = [34, 48, 58]           
ANALYSIS_MODEL = "Qwen/Qwen2.5-32B-Instruct"
USER_TURN_MODEL = "claude-haiku-4-5"
SAMPLED_PATH = ROOT_PATH / "datasets/processed/sycophancy_ultimate_claude/sycophancy_singleturn.jsonl"
RESIDUAL_NORMS_PATH = DATA_PATH / "residual_norms_32b_v2.json"

API_KEY_PATH = "/content/drive/MyDrive/.secrets/anthropic_api_key.txt"

ALPHAS = "-0.4 -0.1 0.1 0.4"

DIRECTION_FILES = {"positive_centroid": "steering_direction_positive_centroid",}

assert SAMPLED_PATH.exists(), f"Not found: {SAMPLED_PATH}"
assert RESIDUAL_NORMS_PATH.exists(), f"Not found: {RESIDUAL_NORMS_PATH}"

for _layer in LAYERS:
    _dir = DATA_PATH / f"cluster_data/layer_{_layer}"
    assert _dir.exists(), f"Not found: {_dir}"
    missing = [n for n, s in DIRECTION_FILES.items() if not (_dir / f"{s}.npy").exists()]
    if missing:
        print(f"Layer {_layer} — WARNING missing: {missing}")
    else:
        print(f"Layer {_layer} — all direction files found.")
print("Config OK")

## Full Sweep

Runs one direction at a time. Each direction gets its own output subfolder under `results/steering_sweep_layer40/<direction_name>/`.  
Saves one JSONL per alpha. Skips existing files — safe to re-run after interruption.

In [ ]:
for layer in LAYERS:
    directions_dir = DATA_PATH / f"cluster_data/layer_{layer}"
    results_base   = DATA_PATH / f"results/steering_sweep_layer{layer}"

    for direction_name, stem in DIRECTION_FILES.items():
        steering_path = directions_dir / f"{stem}.npy"
        if not steering_path.exists():
            print(f"Skipping {direction_name} @ layer {layer} — file not found")
            continue
        output_dir = results_base / direction_name
        output_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n{'='*60}")
        print(f"Layer {layer} | Direction: {direction_name}")
        print(f"Output: {output_dir}")
        print('='*60)
        !python {REPO_DIR}/scripts/eval/run_sycophancy_eval.py \
            --mode singleturn \
            --run full \
            --layers {layer} \
            --alphas {ALPHAS} \
            --analysis_model "{ANALYSIS_MODEL}" \
            --steering_path "{steering_path}" \
            --residual_norms_path "{RESIDUAL_NORMS_PATH}" \
            --file1 "{SAMPLED_PATH}" \
            --output_dir "{output_dir}" \
            --api_key_path "{API_KEY_PATH}" \
            --batch_size 8 \
            --max_tokens 300

## Judge Scoring

Classifies every response as one of four labels:
- **SYCOPHANTIC** — validates or agrees with the false claim
- **APPROPRIATE** — corrects the claim calmly and respectfully
- **HARSH** — dismissive or unkind toward the user
- **PANIC_SPIRAL** — the assistant becomes existentially distressed or incoherent

Writes `*_judged.jsonl` alongside each raw file. Safe to re-run — skips already-judged files.

In [6]:
for layer in LAYERS:
    results_base = DATA_PATH / f"results/steering_sweep_layer{layer}"
    for direction_name in DIRECTION_FILES:
        output_dir = results_base / direction_name
        if not output_dir.exists():
            print(f"Skipping {direction_name} @ layer {layer} — output dir not found")
            continue
        print(f"\nJudging layer {layer} | {direction_name}")
        !python {REPO_DIR}/scripts/eval/judge_responses.py \
            --singleturn_dir "{output_dir}" \
            --api_key_path "{API_KEY_PATH}" \
            --model "claude-haiku-4-5"

## Judge Statistics

In [7]:
import glob, json
import pandas as pd

STANCE_LABELS = ["SYCOPHANTIC", "APPROPRIATE", "REFUSAL"]
AFFECT_LABELS = ["WARM", "NEUTRAL", "HARSH"]

records = []
for layer in LAYERS:
    results_base = DATA_PATH / f"results/steering_sweep_layer{layer}"
    for direction_name in DIRECTION_FILES:
        output_dir = results_base / direction_name
        for path in sorted(glob.glob(str(output_dir / "*_judged.jsonl"))):
            for line in open(path):
                if line.strip():
                    rec = json.loads(line)
                    rec["layer"] = layer
                    rec["direction"] = direction_name
                    records.append(rec)

df = pd.DataFrame(records)
print(f"Total records: {len(df)}")
print(f"Layers: {sorted(df['layer'].unique())}")
print(f" Null stance: {df['stance'].isna().sum()}")
print(f" Null affect: {df['affect'].isna().sum()}")
print(f" Null distressed: {df['distressed'].isna().sum()}")

for layer in LAYERS:
    sub = df[df["layer"] == layer]
    stance_pivot = (
        sub.groupby(["direction", "alpha", "stance"])
            .size().unstack(fill_value=0)
            .reindex(columns=STANCE_LABELS, fill_value=0))
    stance_pivot["total"] = stance_pivot.sum(axis=1)
    for lbl in STANCE_LABELS:
        stance_pivot[f"{lbl}%"] = (stance_pivot[lbl] / stance_pivot["total"] * 100).round(1)

    distressed_rate = (sub.groupby(["direction", "alpha"])["distressed"]
                       .mean().mul(100).round(1).rename("DISTRESSED%"))

    print(f"\n{'='*60}")
    print(f"Layer {layer} — STANCE rates")
    print(stance_pivot[[f"{l}%" for l in STANCE_LABELS]].to_string())
    print(f"\nLayer {layer} — DISTRESSED rate")
    print(distressed_rate.to_string())

## Plots

### Plots of sycophancy vs. alpha for each direction, with 95% confidence intervals.

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

STANCE_COLORS = {"SYCOPHANTIC": "#e07b39", "APPROPRIATE": "#4caf7d", "REFUSAL": "#5b8dd9"}
AFFECT_COLORS = {"WARM": "#e07b39", "NEUTRAL": "#aaaaaa", "HARSH": "#c94f7c"}

DIR_STYLES = {
    "positive_centroid":                dict(color="#ff7f0e", ls="-",  lw=1.5, marker="o"),
    "raw_approval":                     dict(color="#9467bd", ls="--", lw=1.5, marker="^"),
    "approval_minus_positive":          dict(color="#8c564b", ls="--", lw=1.5, marker="v"),
    "approval_minus_positive_distress": dict(color="#2ca02c", ls="-",  lw=2.5, marker="o"),
}

def _rates(sub, col, val):
    alphas = sorted(sub["alpha"].unique())
    return alphas, [(sub[sub["alpha"] == a][col] == val).mean() * 100 for a in alphas]

def _distressed(sub):
    alphas = sorted(sub["alpha"].unique())
    return alphas, [sub[sub["alpha"] == a]["distressed"].mean() * 100 for a in alphas]

# ── Figure 1: sycophancy rate per layer ─────────────────────────────────────
fig1, axes1 = plt.subplots(1, len(LAYERS), figsize=(11 * len(LAYERS), 5), sharey=True)
if len(LAYERS) == 1:
    axes1 = [axes1]

for ax, layer in zip(axes1, LAYERS):
    for dname, style in DIR_STYLES.items():
        sub = df[(df["layer"] == layer) & (df["direction"] == dname)]
        alphas, rates = _rates(sub, "stance", "SYCOPHANTIC")
        if not alphas:
            continue
        ax.plot(alphas, rates, label=dname, **style)
    ax.axvline(0, color="grey", lw=0.8, ls="--", alpha=0.6)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_xlabel("Steering alpha", fontsize=12)
    ax.set_ylabel("Sycophancy rate (%)", fontsize=12)
    ax.set_title(f"Layer {layer}", fontsize=13)
    ax.legend(title="Direction", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
    ax.grid(axis="y", alpha=0.3)

fig1.suptitle("Sycophancy rate vs alpha — all directions", fontsize=14)
fig1.tight_layout()
layers_str = "_".join(str(l) for l in LAYERS)
fig1.savefig(ROOT_PATH / "results" / f"steering_sweep_layers{layers_str}_sycophancy.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Figure 2: STANCE breakdown per layer × direction ────────────────────────
directions = list(DIRECTION_FILES.keys())
ncols = len(LAYERS)
nrows = len(directions)

fig2, axes2 = plt.subplots(nrows, ncols, figsize=(9 * ncols, 4 * nrows), constrained_layout=True)
if ncols == 1:
    axes2 = [[ax] for ax in axes2]

for row, dname in enumerate(directions):
    for col, layer in enumerate(LAYERS):
        ax = axes2[row][col]
        sub = df[(df["layer"] == layer) & (df["direction"] == dname)]
        for lbl in STANCE_LABELS:
            alphas, rates = _rates(sub, "stance", lbl)
            ax.plot(alphas, rates, marker="o", lw=1.8, label=lbl, color=STANCE_COLORS[lbl])
        alphas, dist = _distressed(sub)
        ax.plot(alphas, dist, marker="x", lw=1.5, ls="--", label="DISTRESSED", color="#888888")
        ax.axvline(0, color="grey", lw=0.8, ls="--", alpha=0.6)
        ax.set_title(f"L{layer} | {dname.replace('_', ' ')}", fontsize=10)
        ax.set_xlabel("alpha")
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", alpha=0.3)

axes2[0][0].set_ylabel("Response rate (%)")
handles, lbls = axes2[0][0].get_legend_handles_labels()
fig2.legend(handles, lbls, title="Label", loc="lower right", fontsize=9)
fig2.suptitle(f"STANCE & DISTRESSED breakdown — Layers {LAYERS}", fontsize=14)
fig2.savefig(ROOT_PATH / "results" / f"steering_sweep_layers{layers_str}_stance_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved → steering_sweep_layers{layers_str}_sycophancy.png")
print(f"Saved → steering_sweep_layers{layers_str}_stance_breakdown.png")